In [1]:
from pathlib import Path
import pandas as pd
import re

In [2]:
CSV_DIR = Path("../data/raw/csv")

csv_files = list(CSV_DIR.glob("*.csv"))

print("CSV encontrados:", len(csv_files))

for f in csv_files[:10]:
    print(f.name)

CSV encontrados: 8
group_100_ruminating_rumia.csv
group_102_ruminating_rumia.csv
group_106_ruminating_rumiando.csv
group_1_ruminating_e7413f25-d0c4-44ea-b6f7-f176e7888d30_8666381325437215801.csv
group_20_ruminating.csv
grupo 2 tiempo de rumia.csv
grupo 3 tiempo de rumia.csv
tiempo de rumia.csv


In [3]:
def ExtractMetadataFromFilename(file_path):

    name = file_path.stem

    group_match = re.search(r"group[_\s]*(\d+)", name, re.IGNORECASE)
    collar_match = re.search(r"(\d{8,})$", name)

    group = int(group_match.group(1)) if group_match else None
    collar_id = collar_match.group(1) if collar_match else None

    return group, collar_id

In [4]:
def LoadCsv(file_path):

    try:
        df = pd.read_csv(file_path)
    except:
        df = pd.read_csv(file_path, encoding="latin1")

    group, collar_id = ExtractMetadataFromFilename(file_path)

    df["source_file"] = file_path.name
    df["group_from_file"] = group
    df["collar_from_file"] = collar_id

    return df

In [5]:
dfs = []

for csv_file in csv_files:

    try:
        df = LoadCsv(csv_file)
        dfs.append(df)

    except Exception as e:
        print("Error en:", csv_file.name, e)

rumination_df = pd.concat(dfs, ignore_index=True)

print("Dataset combinado:", rumination_df.shape)

rumination_df.head()

Dataset combinado: (19110, 11)


,VID,EID,LID,Group,Last calving,Days in milk,Date,Ruminating,source_file,group_from_file,collar_from_file
0,1243,NaN,NaN,20.0,29/08/2024,302.0,27/06/2025,NaN,group_100_ruminating_rumia.csv,100,None
1,1243,NaN,NaN,20.0,29/08/2024,303.0,28/06/2025,553.0,group_100_ruminating_rumia.csv,100,None
2,1243,NaN,NaN,20.0,29/08/2024,304.0,29/06/2025,611.0,group_100_ruminating_rumia.csv,100,None
3,1243,NaN,NaN,20.0,29/08/2024,305.0,30/06/2025,646.0,group_100_ruminating_rumia.csv,100,None
4,1243,NaN,NaN,20.0,29/08/2024,306.0,01/07/2025,536.0,group_100_ruminating_rumia.csv,100,None


In [7]:
rumination_df = rumination_df.rename(columns={
    "VID": "cow_id",
    "EID": "eid",
    "LID": "lid",
    "Group": "group_id",
    "Last calving": "last_calving",
    "Days in milk": "days_in_milk",
    "Date": "date",
    "Ruminating": "ruminating_minutes"
})

In [8]:
rumination_df["date"] = pd.to_datetime(
    rumination_df["date"],
    dayfirst=True,
    errors="coerce"
)

rumination_df["last_calving"] = pd.to_datetime(
    rumination_df["last_calving"],
    dayfirst=True,
    errors="coerce"
)

In [9]:
rumination_df["weekday"] = rumination_df["date"].dt.weekday
rumination_df["month"] = rumination_df["date"].dt.month

In [10]:
rumination_df["lactation_age"] = (
    rumination_df["date"] - rumination_df["last_calving"]
).dt.days

In [11]:
rumination_df["resolved_group"] = rumination_df["group_id"].fillna(
    rumination_df["group_from_file"]
)

In [13]:
OUTPUT_PATH = Path("../data/interim/rumination.parquet")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

rumination_df.to_parquet(OUTPUT_PATH)

print("Dataset guardado en:", OUTPUT_PATH)

Dataset guardado en: ..\data\interim\rumination.parquet


In [10]:
import pandas as pd

milking = pd.read_parquet("../data/interim/milking.parquet")
rumination = pd.read_parquet("../data/interim/rumination.parquet")
weather = pd.read_parquet("../data/interim/weather.parquet")
pdf = pd.read_parquet("../data/interim/pdf_events.parquet")

In [11]:
def print_ids(name, df):
    if "cow_id" in df.columns:
        ids = sorted(df["cow_id"].dropna().unique())
        print(f"\n{name} ({len(ids)} cows)")
        print(ids)

print_ids("MILKING", milking)
print_ids("RUMINATION", rumination)
print_ids("PDF EVENTS", pdf)


MILKING (65 cows)
[np.int64(1204), np.int64(1213), np.int64(1216), np.int64(1225), np.int64(1228), np.int64(1235), np.int64(1236), np.int64(1242), np.int64(1243), np.int64(1497), np.int64(1510), np.int64(1555), np.int64(1560), np.int64(1567), np.int64(1577), np.int64(1590), np.int64(1617), np.int64(1638), np.int64(1644), np.int64(2070), np.int64(2074), np.int64(2076), np.int64(2082), np.int64(2087), np.int64(2108), np.int64(2110), np.int64(2119), np.int64(2122), np.int64(2150), np.int64(2165), np.int64(5767), np.int64(5981), np.int64(6012), np.int64(6050), np.int64(6054), np.int64(6062), np.int64(6070), np.int64(6092), np.int64(6098), np.int64(6124), np.int64(6131), np.int64(6134), np.int64(6137), np.int64(6154), np.int64(6161), np.int64(6164), np.int64(6178), np.int64(6193), np.int64(6194), np.int64(6197), np.int64(6226), np.int64(6258), np.int64(8703), np.int64(8705), np.int64(8707), np.int64(8708), np.int64(8710), np.int64(8711), np.int64(8712), np.int64(8714), np.int64(8715), np.i

In [12]:
for name, df in {
    "MILKING": milking,
    "RUMINATION": rumination,
    "PDF": pdf
}.items():
    print("\n", name)
    for col in df.columns:
        if "id" in col.lower():
            print(col)


 MILKING
cow_id

 RUMINATION
cow_id
eid
lid
group_id

 PDF
animal_id
